# BaseballCV — Base Detector Example

This notebook tests the **base detector** integration end-to-end:

- `base_detector` model (YOLOv11s, detects bases in broadcast feeds)
- `base_detect_dataset` (YOLO-format dataset)
- `base_detector_coco` (COCO-format dataset)

All three are resolved through `LoadTools` aliases and downloaded from the BallDataLab vault.

> **Note:** requires `baseballcv` 0.1.25 or later ([PR #179](https://github.com/BaseballCV/BaseballCV/pull/179)).

**Runtime:** `Runtime → Change runtime type → GPU` recommended (CPU works, just slower).

In [ ]:
# Install BaseballCV (base detector requires >= 0.1.25)
!pip install -q "baseballcv>=0.1.25"

In [ ]:
import os
import json
import urllib.request
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

from baseballcv.functions import LoadTools

load_tools = LoadTools()
print("Setup OK")

## 1. Load the model

`load_model` resolves the alias through the BallDataLab API (`balldatalab.com/api/models/base_detector`), which redirects to the vault-hosted weights.

In [ ]:
model_path = load_tools.load_model('base_detector')
print("Model weights at:", model_path)

model = YOLO(model_path)
print("Task:", model.task, "| Classes:", model.names)

## 2. Inference on a broadcast example

A real broadcast frame from the repo's assets. The model has a single class (`base`); on real-world frames it can be conservative, so we use a low confidence threshold.

In [ ]:
IMG_URL = "https://raw.githubusercontent.com/BaseballCV/BaseballCV/main/assets/Tatis_SB.jpg"
img_path = "example.jpg"
urllib.request.urlretrieve(IMG_URL, img_path)

result = model.predict(img_path, conf=0.1, verbose=False)[0]

plt.figure(figsize=(12, 6))
plt.imshow(result.plot()[:, :, ::-1])  # plot() returns BGR
plt.axis("off")
plt.title(f"base_detector — {len(result.boxes)} detection(s) at conf >= 0.1")
plt.show()

## 3. COCO dataset — `base_detector_coco`

Downloads and extracts the COCO-format dataset (~150 MB): 660 images, 668 annotations, single class `base` (id `0`).

In [ ]:
coco_dir = load_tools.load_dataset('base_detector_coco')
print("Dataset extracted to:", coco_dir)

In [ ]:
ann_file = os.path.join(str(coco_dir), "annotations", "instances_default.json")
with open(ann_file) as f:
    coco = json.load(f)

cats = {c["id"]: c["name"] for c in coco["categories"]}
dist = Counter(cats[a["category_id"]] for a in coco["annotations"])

print("Images:", len(coco["images"]))
print("Annotations:", len(coco["annotations"]))
print("Categories:", cats)
print("Class distribution:", dict(dist))

### Ground truth vs. model prediction

Side-by-side on a random annotated image (the model was trained on these, so detections should match well here — this is a pipeline test, not a benchmark).

In [ ]:
img_info = next(im for im in coco["images"]
                if any(a["image_id"] == im["id"] for a in coco["annotations"]))
sample_path = os.path.join(str(coco_dir), "images", img_info["file_name"])
gt = [a for a in coco["annotations"] if a["image_id"] == img_info["id"]]

pred = model.predict(sample_path, conf=0.25, verbose=False)[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(Image.open(sample_path))
axes[0].set_title(f"Ground truth — {len(gt)} box(es)")
for a in gt:
    x, y, w, h = a["bbox"]
    axes[0].add_patch(plt.Rectangle((x, y), w, h, fill=False, color="lime", linewidth=2))
axes[1].imshow(pred.plot()[:, :, ::-1])
axes[1].set_title(f"base_detector — {len(pred.boxes)} detection(s)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. YOLO dataset — `base_detect_dataset`

Same 660 images in YOLO format. Known issue (documented in the dataset README): 107 of the 660 images have no label file yet, and there is no val/test split.

In [ ]:
yolo_dir = load_tools.load_dataset('base_detect_dataset')
print("Dataset extracted to:", yolo_dir)

with open(os.path.join(str(yolo_dir), "data.yaml")) as f:
    print("--- data.yaml ---")
    print(f.read())

n_imgs = sum(len(fs) for _, _, fs in os.walk(os.path.join(str(yolo_dir), "images")))
n_lbls = sum(len(fs) for _, _, fs in os.walk(os.path.join(str(yolo_dir), "labels")))
print(f"images: {n_imgs} | label files: {n_lbls} | unlabeled: {n_imgs - n_lbls}")

## Wrap-up

If everything above ran cleanly, the full integration works:

| Asset | Alias | Status |
|---|---|---|
| Base detector model | `base_detector` | ✅ loads via BDL API |
| COCO dataset | `base_detector_coco` | ✅ downloads + extracts |
| YOLO dataset | `base_detect_dataset` | ✅ downloads + extracts |

Next steps: resolve the 107 unlabeled images, add a val split + metrics.